# freehiero — Detector 測試

比較 **KeywordDetector**（L1）與 **Qwen2.5-0.5B**（L2）在偵測「免費食物」貼文的準確率。

執行環境：Colab（T4 GPU）

## 1. 安裝依賴

In [ ]:
!pip install -q transformers accelerate

## 2. KeywordDetector（直接複製 detector.py 的 L1 邏輯）

In [ ]:
import re

_FREE_WORDS = r'免費|free|請拿|拿走|多餘|多的|送人|不要了|剩食|剩菜|拿去|有需要|自取|帶走|送出|分享'
_FOOD_WORDS = r'食物|食品|飯|麵|便當|零食|餅乾|水果|蔬菜|菜|湯|肉|蛋|麵包|吐司|料理|點心|糕|餅|粽|飲料|奶茶|咖啡|茶|寶特瓶'
_PATTERN_FREE_FOOD = re.compile(rf'(?=.*({_FREE_WORDS}))(?=.*({_FOOD_WORDS}))', re.IGNORECASE)
_PATTERN_NOT_FOOD  = re.compile(r'免費.*?(?:課程|諮詢|活動|講座|workshop|票|名額|參加|索取)', re.IGNORECASE)

def keyword_detect(text: str) -> bool:
    if not text:
        return False
    if _PATTERN_NOT_FOOD.search(text):
        return False
    return bool(_PATTERN_FREE_FOOD.search(text))

print('KeywordDetector loaded')

## 3. 測試資料集

**標記說明**：`1` = 確實在送免費食物，`0` = 非食物或不是免費送

> 上線前請補充真實社團貼文（正例 + 負例各 ≥ 15 篇）

In [ ]:
SAMPLES = [
    # ── 正例（label=1）────────────────────────────────────────────
    (1, '實驗室訂太多便當，剩3個，有需要的同學來工程館205自取！'),
    (1, '有多的蛋糕，不要浪費，歡迎拿走，在宿舍A棟1F公共桌'),
    (1, '家裡帶了一大袋水果，拿不完，放在系辦門口，自己拿'),
    (1, '免費送！昨天活動剩下的麵包，大概10個，先到先拿'),
    (1, '多餘的泡麵一箱，有人要嗎？在圖書館一樓大廳'),
    (1, '零食太多吃不完，各種餅乾飲料，送給有需要的人，WP大廳'),
    (1, '今天煮太多了，剩一鍋湯，想送人，有人要來拿嗎？台南市區'),
    (1, '有幾袋茶葉蛋，不要了，免費拿，在生科館門口'),
    (1, '寶特瓶裝的黑糖奶茶 6瓶 送人 宿舍B棟大廳'),
    (1, 'free food！活動剩下三明治跟沙拉，在學生活動中心'),
    # ── 負例（label=0）────────────────────────────────────────────
    (0, '二手教科書出售，工程數學，9成新，200元'),
    (0, '免費諮詢！心理輔導中心開放預約，歡迎同學來談'),
    (0, '有人要一起組隊打羽毛球嗎？每週三晚上'),
    (0, '請問有人知道哪裡可以修腳踏車嗎？'),
    (0, '免費活動！社團公演，歡迎來看，不用報名'),
    (0, '出售 Switch遊戲片，便宜賣'),
    (0, '有人有多的洗衣精嗎？可以換東西'),
    (0, '徵室友，套房，近學校，水電費另計'),
    (0, '教科書出售，普通化學，保存良好，250元有意者私訊'),
    (0, '今天天氣很好，大家出去走走吧'),
]

print(f'資料集：{sum(l==1 for l,_ in SAMPLES)} 正例 / {sum(l==0 for l,_ in SAMPLES)} 負例')

## 4. 評估 KeywordDetector

In [ ]:
def evaluate(name, predict_fn, samples):
    tp = fp = tn = fn = 0
    errors = []
    for label, text in samples:
        pred = predict_fn(text)
        if label == 1 and pred:     tp += 1
        elif label == 0 and not pred: tn += 1
        elif label == 0 and pred:
            fp += 1
            errors.append(('FP', text[:60]))
        else:
            fn += 1
            errors.append(('FN', text[:60]))

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall    = tp / (tp + fn) if (tp + fn) else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0

    print(f'\n── {name} ──')
    print(f'  Precision: {precision:.2%}  Recall: {recall:.2%}  F1: {f1:.2%}')
    print(f'  TP={tp} FP={fp} TN={tn} FN={fn}')
    if errors:
        print('  錯誤案例:')
        for tag, t in errors:
            print(f'    [{tag}] {t}')
    return dict(name=name, precision=precision, recall=recall, f1=f1)

kw_result = evaluate('KeywordDetector (L1)', keyword_detect, SAMPLES)

## 5. Qwen2.5-0.5B-Instruct（L2 模型，需 GPU）

模擬 `OllamaDetector` 的邏輯，但改用 HuggingFace Transformers 直接跑。
這樣不需要 Ollama server，在 Colab 上也能驗證模型品質。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)
print(f'模型載入完成，裝置：{next(model.parameters()).device}')

In [ ]:
PROMPT_TMPL = (
    '你是食物偵測助理。判斷以下社團貼文是否在免費提供食物（讓人拿取）。'
    '只回答 yes 或 no，不要解釋。\n\n貼文：{text}'
)

def qwen_detect(text: str) -> bool:
    messages = [
        {'role': 'system', 'content': '你只回答 yes 或 no。'},
        {'role': 'user', 'content': PROMPT_TMPL.format(text=text[:400])},
    ]
    ids = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=5, do_sample=False)
    answer = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip().lower()
    return answer.startswith('yes')

# Quick smoke test
print(qwen_detect('有多的便當，免費拿走，在工程館'))  # expect True
print(qwen_detect('出售二手書'))                       # expect False

In [ ]:
qwen_result = evaluate('Qwen2.5-0.5B (L2)', qwen_detect, SAMPLES)

## 6. 比較結果

In [ ]:
print(f'\n{'模型':<25} {'Precision':>10} {'Recall':>10} {'F1':>10}')
print('-' * 58)
for r in [kw_result, qwen_result]:
    print(f"{r['name']:<25} {r['precision']:>10.2%} {r['recall']:>10.2%} {r['f1']:>10.2%}")

## 7. 兩層合併效果（L1 miss → L2）

In [ ]:
_ANY_SIGNAL = re.compile(rf'({_FREE_WORDS})|({_FOOD_WORDS})', re.IGNORECASE)

def two_layer_detect(text: str) -> bool:
    if keyword_detect(text):
        return True
    # only send to model if there's at least some signal
    if _ANY_SIGNAL.search(text):
        return qwen_detect(text)
    return False

two_result = evaluate('TwoLayer (L1 + L2)', two_layer_detect, SAMPLES)

print(f'\n{'模型':<25} {'Precision':>10} {'Recall':>10} {'F1':>10}')
print('-' * 58)
for r in [kw_result, qwen_result, two_result]:
    print(f"{r['name']:<25} {r['precision']:>10.2%} {r['recall']:>10.2%} {r['f1']:>10.2%}")

## 8. 新增自訂測試樣本

把遇到的真實貼文貼進來，看哪層漏掉、原因為何。

In [ ]:
CUSTOM = [
    # 貼上真實貼文測試，格式：(true_label, '貼文內容')
    # (1, '...'),
    # (0, '...'),
]

if CUSTOM:
    print('=== KeywordDetector ===')
    evaluate('KeywordDetector', keyword_detect, CUSTOM)
    print('\n=== Qwen2.5 ===')
    evaluate('Qwen2.5', qwen_detect, CUSTOM)
    print('\n=== TwoLayer ===')
    evaluate('TwoLayer', two_layer_detect, CUSTOM)
else:
    print('CUSTOM 為空，請填入真實貼文後重新執行')